In [5]:
import sys
sys.path.append('/gpfs/work/aac/xingyiyao23/Code')
import nibabel as nb
import glob
import os
import glob
import lpips
import torch
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

import Diffusion_denoising_thin_slice.functions_collection as ff
import Diffusion_denoising_thin_slice.Build_lists.Build_list as Build_list
import Diffusion_denoising_thin_slice.Data_processing as Data_processing

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
build_sheet =  Build_list.Build_thinsliceCT(os.path.join('/gpfs/work/aac/xingyiyao23/Data/PCCT/Patient_lists/PCCT_split.xlsx'))
_,patient_id_list,patient_subid_list,random_num_list, condition_list, x0_list = build_sheet.__build__(batch_list = [2]) 
n = ff.get_X_numbers_in_interval(total_number = patient_id_list.shape[0],start_number = 0,end_number = 1, interval = 1)


In [16]:
def calculate_CNR(img,wm_roi, gm_roi):
    wm_mean = np.mean(img[wm_roi==1])
    wm_std = np.std(img[wm_roi==1])
    gm_mean = np.mean(img[gm_roi==1])
    gm_std = np.std(img[gm_roi==1])
    contrast = gm_mean - wm_mean
    contrast_to_noise_ratio = (gm_mean - wm_mean) / np.sqrt(gm_std**2 + wm_std**2)
    
    return wm_mean, wm_std, gm_mean, gm_std, contrast, contrast_to_noise_ratio

## CNR

In [17]:
## metric calculations
import shutil

results = []
for i in range(0,n.shape[0]):
    patient_id = patient_id_list[n[i]]

    # original image
    original_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT_nohist/pred_images', patient_id, 'epoch130_1/condition_img.nii.gz')
    condition_img = nb.load(original_file).get_fdata()
    # turn pixels not in [0,100] to 0 or 100
    condition_img[condition_img < 0] = 0
    condition_img[condition_img > 100] = 100
    

    # unsupervised
    unsupervised_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT_nohist/pred_images', patient_id, 'epoch130avg/pred_img_scans8.nii.gz')
    unsupervised_img = nb.load(unsupervised_file).get_fdata()
    unsupervised_img[unsupervised_img < 0] = 0
    unsupervised_img[unsupervised_img > 100] = 100

    # noise2noise
    noise2noise_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/noise2noise_PCCT/pred_images', patient_id, 'epoch_80/pred_img.nii.gz')
    noise2noise_img = nb.load(noise2noise_file).get_fdata() 
    noise2noise_img[noise2noise_img < 0] = 0
    noise2noise_img[noise2noise_img > 100] = 100

    # ddm2 first step
    ddm2_firststep_file = os.path.join('/host/d/projects/denoising/pred_collections/PCCT/', patient_id, 'ddm2_firststep_image.nii.gz')
    ddm2_first_img = nb.load(ddm2_firststep_file).get_fdata()
    ddm2_first_img[ddm2_first_img < 0] = 0
    ddm2_first_img[ddm2_first_img > 100] = 100

    # white matter ROI
    wm_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT_nohist/pred_images', patient_id, 'WM_ROI.nii.gz')
    wm_roi = nb.load(wm_file).get_fdata(); wm_roi = np.round(wm_roi)

    # grey matter ROI
    gm_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT_nohist/pred_images', patient_id, 'GM_ROI.nii.gz')
    gm_roi = nb.load(gm_file).get_fdata(); gm_roi = np.round(gm_roi)

    # calculated CNR
    wm_mean_condition, wm_std_condition, gm_mean_condition, gm_std_condition, contrast_condition, cnr_condition = calculate_CNR(condition_img, wm_roi, gm_roi)
    print('condition: wm_mean', wm_mean_condition, 'wm_std', wm_std_condition, 'gm_mean', gm_mean_condition, 'gm_std', gm_std_condition, 'CNR', cnr_condition)
    wm_mean_n2n, wm_std_n2n, gm_mean_n2n, gm_std_n2n, contrast_n2n, cnr_n2n = calculate_CNR(noise2noise_img, wm_roi, gm_roi)
    print('noise2noise: wm_mean', wm_mean_n2n, 'wm_std', wm_std_n2n, 'gm_mean', gm_mean_n2n, 'gm_std', gm_std_n2n, 'CNR', cnr_n2n)
    wm_mean_ddm2, wm_std_ddm2, gm_mean_ddm2, gm_std_ddm2, contrast_ddm2, cnr_ddm2 = calculate_CNR(ddm2_first_img, wm_roi, gm_roi)
    print('ddm2 first step: wm_mean', wm_mean_ddm2, 'wm_std', wm_std_ddm2, 'gm_mean', gm_mean_ddm2, 'gm_std', gm_std_ddm2, 'CNR', cnr_ddm2)
    wm_mean_unsupervised, wm_std_unsupervised, gm_mean_unsupervised, gm_std_unsupervised, contrast_unsupervised, cnr_unsupervised = calculate_CNR(unsupervised_img, wm_roi, gm_roi)
    print('unsupervised: wm_mean', wm_mean_unsupervised, 'wm_std', wm_std_unsupervised, 'gm_mean', gm_mean_unsupervised, 'gm_std', gm_std_unsupervised, 'CNR', cnr_unsupervised)

    print('Patient ID: ', patient_id, 'CNR Condition: ', cnr_condition, 'CNR N2N: ', cnr_n2n, 'CNR DDM2 first step: ', cnr_ddm2, 'CNR Unsupervised: ', cnr_unsupervised)

    results.append([patient_id,
                    wm_mean_condition, wm_std_condition, gm_mean_condition, gm_std_condition, contrast_condition, cnr_condition,
                    wm_mean_n2n, wm_std_n2n, gm_mean_n2n, gm_std_n2n, contrast_n2n, cnr_n2n,
                    wm_mean_ddm2, wm_std_ddm2, gm_mean_ddm2, gm_std_ddm2, contrast_ddm2, cnr_ddm2,
                    wm_mean_unsupervised, wm_std_unsupervised, gm_mean_unsupervised, gm_std_unsupervised, contrast_unsupervised, cnr_unsupervised])
    dd = pd.DataFrame(results, columns = ['Patient_ID',
                                          'WM_Mean_Condition', 'WM_Std_Condition', 'GM_Mean_Condition', 'GM_Std_Condition', 'Contrast_Condition', 'CNR_Condition',
                                          'WM_Mean_N2N', 'WM_Std_N2N', 'GM_Mean_N2N', 'GM_Std_N2N', 'Contrast_N2N', 'CNR_N2N',
                                          'WM_Mean_DDM2', 'WM_Std_DDM2', 'GM_Mean_DDM2', 'GM_Std_DDM2', 'Contrast_DDM2', 'CNR_DDM2',
                                          'WM_Mean_Unsupervised', 'WM_Std_Unsupervised', 'GM_Mean_Unsupervised', 'GM_Std_Unsupervised', 'Contrast_Unsupervised', 'CNR_Unsupervised'])
    dd.to_excel('/host/d/projects/denoising/results/PCCT_CNR.xlsx', index = False)

condition: wm_mean 21.166900420757365 wm_std 10.190137605442528 gm_mean 30.62719298245614 gm_std 9.240858784118547 CNR 0.6877123866485128
noise2noise: wm_mean 25.052918232771344 wm_std 3.1715091198631917 gm_mean 32.30924954314239 gm_std 3.229911268172923 CNR 1.603015343310427
ddm2 first step: wm_mean 22.192847124824684 wm_std 7.176885868666201 gm_mean 29.36842105263158 gm_std 9.105004559759484 CNR 0.6189317949466818
unsupervised: wm_mean 18.849263066830844 wm_std 3.655004635426696 gm_mean 29.87331407111988 gm_std 4.399204448360321 CNR 1.9274684398901354
Patient ID:  29 CNR Condition:  0.6877123866485128 CNR N2N:  1.603015343310427 CNR DDM2 first step:  0.6189317949466818 CNR Unsupervised:  1.9274684398901354
condition: wm_mean 21.734964322120284 wm_std 12.736932530207314 gm_mean 30.951015531660694 gm_std 11.386287401286909 CNR 0.5394422387177898
noise2noise: wm_mean 25.521404508622364 wm_std 3.7800661365415364 gm_mean 31.584596001485913 gm_std 3.655187590538517 CNR 1.153079479175441
dd

## SNR

In [10]:
## metric calculations
from genericpath import isfile
import shutil

results = []
save_collection = True
for i in range(0,n.shape[0]):
    patient_id = patient_id_list[n[i]]

    if patient_id == '29':
        center = [315,249]
    elif patient_id == '30':
        center = [275,188]
    elif patient_id == '31':
        center = [233,253]
    elif patient_id == '32':
        center = [286,236]
    elif patient_id == '33':
        center = [281,217]
    elif patient_id == '34':
        center = [200,191]
    elif patient_id == '35':
        center = [331,269]
    elif patient_id == '36':
        center = [294,288]
    # original image
    original_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT/pred_images', patient_id, 'epoch118_1/condition_img.nii.gz')
    condition_img = nb.load(original_file).get_fdata()
    # turn pixels not in [0,100] to 0 or 100
    condition_img[condition_img < 0] = 0
    condition_img[condition_img > 100] = 100

    # unsupervised
    unsupervised_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/unsupervised_gaussian_PCCT/pred_images', patient_id, 'epoch118avg/pred_img_scans20.nii.gz')
    unsupervised_img = nb.load(unsupervised_file).get_fdata()
    unsupervised_img[unsupervised_img < 0] = 0
    unsupervised_img[unsupervised_img > 100] = 100

    # noise2noise
    noise2noise_file = os.path.join('/gpfs/work/aac/xingyiyao23/results/noise2noise_PCCT/pred_images', patient_id, 'epoch_80/pred_img.nii.gz')
    noise2noise_img = nb.load(noise2noise_file).get_fdata() 
    noise2noise_img[noise2noise_img < 0] = 0
    noise2noise_img[noise2noise_img > 100] = 100

    # ddm2 first step
    ddm2_firststep_file = os.path.join('/host/d/projects/denoising/pred_collections/PCCT/', patient_id, 'ddm2_firststep_image.nii.gz')
    ddm2_first_img = nb.load(ddm2_firststep_file).get_fdata()
    ddm2_first_img[ddm2_first_img < 0] = 0
    ddm2_first_img[ddm2_first_img > 100] = 100

    # calculated metrics
    vmin = 0
    vmax = 100
    roi = [center[0]-30, center[0]+30, center[1]-30, center[1]+30]

    # calculate SNR
    mean_condition = np.mean(condition_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    std_condition = np.std(condition_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    snr_condition = mean_condition / std_condition

    mean_n2n = np.mean(noise2noise_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    std_n2n = np.std(noise2noise_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    snr_n2n = mean_n2n / std_n2n

    mean_ddm2_first = np.mean(ddm2_first_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    std_ddm2_first = np.std(ddm2_first_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    snr_ddm2_first = mean_ddm2_first / std_ddm2_first

    mean_unsupervised = np.mean(unsupervised_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    std_unsupervised = np.std(unsupervised_img[roi[0]:roi[1], roi[2]:roi[3], 20:30])
    snr_unsupervised = mean_unsupervised / std_unsupervised

    print('Patient ID: ', patient_id, 'SNR Condition: ', snr_condition, 'SNR N2N: ', snr_n2n, 'SNR DDM2 first step: ', snr_ddm2_first, 'SNR Unsupervised: ', snr_unsupervised)


    results.append([patient_id, snr_condition, snr_n2n, snr_ddm2_first, snr_unsupervised])
    dd = pd.DataFrame(results, columns = ['Patient_ID', 'SNR_Condition', 'SNR_N2N', 'SNR_DDM2_firststep', 'SNR_Unsupervised'])
    dd.to_excel('/host/d/projects/denoising/results/PCCT_SNR.xlsx', index = False)

Patient ID:  29 SNR Condition:  1.3439333721635927 SNR N2N:  3.5220868694260568 SNR DDM2 first step:  1.5216368330661236 SNR Unsupervised:  3.924356146697516
Patient ID:  30 SNR Condition:  1.7166906263469297 SNR N2N:  5.067879657563622 SNR DDM2 first step:  3.84657251596975 SNR Unsupervised:  4.9665673192990205
Patient ID:  31 SNR Condition:  1.6806550679627945 SNR N2N:  5.8786234147333305 SNR DDM2 first step:  2.132537149097087 SNR Unsupervised:  6.000705944092054
Patient ID:  32 SNR Condition:  1.5657230301814007 SNR N2N:  5.4331270729504215 SNR DDM2 first step:  4.959465426048408 SNR Unsupervised:  6.199433998219258
Patient ID:  33 SNR Condition:  1.4905905953887744 SNR N2N:  4.35047841826996 SNR DDM2 first step:  1.3708553711801652 SNR Unsupervised:  5.165028334075216
Patient ID:  34 SNR Condition:  1.2667110021334922 SNR N2N:  2.569164482260312 SNR DDM2 first step:  1.7325921805725233 SNR Unsupervised:  3.0675854603841826
Patient ID:  35 SNR Condition:  0.991425853485143 SNR N2N: